# TUS-REC Real-Data: Train → Evaluate → Validate

Trains the pose/motion network on **real TUS-REC freehand-ultrasound sweeps**, then scores it with the
challenge-style reconstruction-error metrics on a held-out subject split.

**Expected data layout** (attach via **+ Add Data**):
```
TUS-REC-2024/
├── 000/
│   ├── LH_Par_C_DtP.h5      # each .h5: datasets 'frames' (N,H,W), 'tforms' (N,4,4)
│   ├── LH_Par_C_PtD.h5
│   └── ... (hand × orientation × trajectory × sweep-direction)
├── 001/
└── ...
```

**Pipeline:** frame → encoder (ViT/ViG, from the repo) → pose regressor → per-frame relative transform.
Supervised by the GT `tforms` via a point-based distance loss (predict the motion *between* frames; the
tracker poses are used only as ground truth — never as network input, keeping it trackerless).

**Self-contained:** the real-data reader, SE(3) math, loss, and metrics live in this notebook so it runs
against the attached data on any branch. It imports only the repo's `build_encoder` / `build_pose_regressor`.

> **Note on units:** reconstruction error is reported in the tracker's world units (mm) using the GT
> `tforms` plus a pixel→mm scale from `PIXEL_SPACING`. If the official calibration matrix is attached, point
> `CALIB_PATH` at it; otherwise the scale is approximate but the comparison (pred vs GT) is still valid —
> an exact prediction scores 0 regardless.

## 0. Clone + install (encoder/pose come from the repo)

In [ ]:
import os, sys, site, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Armstrong66/2D-to-3D-medical-image-reconstruction.git"
REPO_DIR = "2D-to-3D-medical-image-reconstruction"
BRANCH   = "feat/real-data-loader-eval-metrics"   # or "main"

base = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(base)
if not os.path.isdir(REPO_DIR):
    os.system(f"git clone -q --branch {BRANCH} {REPO_URL} || git clone -q {REPO_URL}")
os.chdir(REPO_DIR)
os.system(f"git checkout -q {BRANCH} 2>/dev/null || true")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "h5py", "-q"], check=False)
site.main()
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "src" / "usrecon").exists():
        if str(cand / "src") not in sys.path:
            sys.path.insert(0, str(cand / "src"))
        break
for m in [k for k in list(sys.modules) if k == "usrecon" or k.startswith("usrecon.")]:
    del sys.modules[m]
print("cwd:", os.getcwd())

## 1. Configuration

In [ ]:
import torch

# ---- Data ----
# Folder under /kaggle/input that contains the numeric subject dirs (000, 001, ...).
# Leave "" to auto-detect the first attached folder that contains .h5 scans.
DATASET_SLUG = ""
CALIB_PATH   = ""          # optional path to an official calibration matrix (.csv/.h5); "" = approximate
LANDMARK_DIR = ""          # optional folder with landmark_XXX.h5 files; "" = skip landmark metrics

# ---- Model / training ----
IMAGE_SIZE   = 64          # frames are resized to IMAGE_SIZE x IMAGE_SIZE
ENCODER_TYPE = "vit"       # "vit" | "vig"
EMBED_DIM    = 128
WINDOW       = 8           # frames per training window (relative motion is learned within it)
EPOCHS       = 3
LR           = 3e-4
MAX_TRAIN_SUBJECTS = 8     # cap for a time-boxed Kaggle run; raise to use more
MAX_VAL_SUBJECTS   = 2
MAX_EVAL_FRAMES    = 128   # subsample long scans at eval to bound cost
PIXEL_SPACING = 0.5        # mm per pixel (used if no calibration file)
METRIC_STRIDE = 8          # pixel grid stride for the all-pixel reconstruction error
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print("device:", DEVICE, "| torch:", torch.__version__)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Locate data + **peek** at the real HDF5 schema

In [ ]:
import h5py, numpy as np
from pathlib import Path

def find_root():
    if DATASET_SLUG:
        p = Path("/kaggle/input") / DATASET_SLUG
        if p.exists():
            return p
    ki = Path("/kaggle/input")
    if ki.exists():
        # prefer a folder whose children are numeric subject dirs holding .h5 files
        for cand in sorted(ki.rglob("*")):
            if cand.is_dir() and any(c.is_dir() and any(c.glob("*.h5")) for c in cand.iterdir()):
                return cand
        for cand in sorted(ki.iterdir()):
            if cand.is_dir() and any(cand.rglob("*.h5")):
                return cand
    return None

DATA_ROOT = find_root()
assert DATA_ROOT is not None, "No attached dataset with .h5 files found under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

subjects = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f"{len(subjects)} subject folders:", subjects[:6], "..." if len(subjects) > 6 else "")

# --- peek: print keys + shapes of the first scan so schema assumptions are visible ---
first_scan = next(iter(sorted((DATA_ROOT / subjects[0]).glob("*.h5"))))
print("\npeeking:", first_scan.name)
with h5py.File(first_scan, "r") as f:
    for k in f.keys():
        print(f"  {k:12s} shape={f[k].shape} dtype={f[k].dtype}")

# adapt to whatever the frames/tforms keys are actually called
with h5py.File(first_scan, "r") as f:
    keys = list(f.keys())
FRAMES_KEY = "frames" if "frames" in keys else keys[0]
TFORMS_KEY = "tforms" if "tforms" in keys else ("tforms" if "tforms" in keys else keys[-1])
print("\nusing FRAMES_KEY =", FRAMES_KEY, "| TFORMS_KEY =", TFORMS_KEY)
print("If those are wrong, set them by hand from the keys printed above.")

## 3. SE(3) math, loss, and reconstruction-error metrics (inline)

In [ ]:
import torch

def quat_trans_to_matrix(p7):
    """(...,7) [tx,ty,tz,qw,qx,qy,qz] -> (...,4,4). Quaternion forced unit-norm."""
    t = p7[..., 0:3]; q = p7[..., 3:7]
    q = q / (q.norm(dim=-1, keepdim=True) + 1e-8)
    w, x, y, z = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
    R = torch.stack([
        1 - 2*(y*y+z*z), 2*(x*y-w*z),     2*(x*z+w*y),
        2*(x*y+w*z),     1 - 2*(x*x+z*z), 2*(y*z-w*x),
        2*(x*z-w*y),     2*(y*z+w*x),     1 - 2*(x*x+y*y),
    ], dim=-1).reshape(*p7.shape[:-1], 3, 3)
    T = torch.zeros(*p7.shape[:-1], 4, 4, dtype=p7.dtype, device=p7.device)
    T[..., :3, :3] = R; T[..., :3, 3] = t; T[..., 3, 3] = 1.0
    return T

def invert_transform(T):
    R = T[..., :3, :3]; t = T[..., :3, 3]; Rt = R.transpose(-1, -2)
    out = torch.zeros_like(T); out[..., :3, :3] = Rt
    out[..., :3, 3] = -(Rt @ t.unsqueeze(-1)).squeeze(-1); out[..., 3, 3] = 1.0
    return out

def relative_from_absolute(tf):
    """(N,4,4) absolute -> (N,4,4) rel[k]=inv(tf[k-1])@tf[k], rel[0]=I."""
    N = tf.shape[0]
    rel = torch.eye(4, dtype=tf.dtype, device=tf.device).repeat(N, 1, 1)
    if N > 1:
        rel[1:] = invert_transform(tf[:-1]) @ tf[1:]
    return rel

def accumulate_global(rel):
    N = rel.shape[0]
    g = torch.eye(4, dtype=rel.dtype, device=rel.device).repeat(N, 1, 1)
    for k in range(1, N):
        g[k] = g[k-1] @ rel[k]
    return g

def accumulate_local(rel, window=1):
    N = rel.shape[0]
    out = torch.eye(4, dtype=rel.dtype, device=rel.device).repeat(N, 1, 1)
    for k in range(N):
        acc = torch.eye(4, dtype=rel.dtype, device=rel.device)
        for j in range(max(1, k-window+1), k+1):
            acc = acc @ rel[j]
        out[k] = acc
    return out

def pixel_to_mm_matrix(spacing, device="cpu"):
    M = torch.eye(4, device=device)
    M[0, 0] = spacing; M[1, 1] = spacing
    return M

def load_calibration(csv_path):
    """Parse the official calib_matrix.csv -> (S, C) 4x4 numpy arrays.
    S = scaling_from_pixel_to_mm ; C = image-coordinate -> tracking-tool transform.
    Robust to the file's two labelled 4x4 blocks."""
    rows = []
    for line in Path(csv_path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            vals = [float(x) for x in line.split(",")]
        except ValueError:
            continue          # a label line
        if len(vals) == 4:
            rows.append(vals)
    arr = np.asarray(rows, dtype=np.float32)
    assert arr.shape[0] >= 8, f"expected two 4x4 blocks, got {arr.shape}"
    return arr[:4], arr[4:8]   # S, C

def build_pixel_to_tool(calib_path, spacing, device):
    """Return (pix->tool 4x4, message, is_calibrated).
    Calibrated: C @ S places an image pixel into tracking-tool mm coordinates.
    Fallback: isotropic pixel spacing with no image->tool offset."""
    if calib_path and Path(calib_path).exists():
        S, C = load_calibration(calib_path)
        S = torch.from_numpy(S).float(); C = torch.from_numpy(C).float()
        M = (C @ S).to(device)
        off = float(torch.linalg.norm(C[:3, 3]))
        msg = (f"CALIBRATED: S=diag({S[0,0]:.4f},{S[1,1]:.4f}) mm/px, "
               f"image->tool offset ~{off:.1f} mm")
        return M, msg, True
    M = pixel_to_mm_matrix(spacing, device)
    return M, f"UNCALIBRATED: isotropic {spacing} mm/px, no image->tool offset", False

def frame_points(H, W, pix2mm, stride=None, device="cpu"):
    """Homogeneous mm points of a frame. corners+center if stride is None, else a grid."""
    if stride is None:
        us = torch.tensor([0, W-1, 0, W-1, (W-1)/2.0], dtype=torch.float32)
        vs = torch.tensor([0, 0, H-1, H-1, (H-1)/2.0], dtype=torch.float32)
    else:
        gu = torch.arange(0, W, stride, dtype=torch.float32)
        gv = torch.arange(0, H, stride, dtype=torch.float32)
        uu, vv = torch.meshgrid(gu, gv, indexing="xy")
        us = uu.reshape(-1); vs = vv.reshape(-1)
    pts = torch.stack([us, vs, torch.zeros_like(us), torch.ones_like(us)], dim=0).to(device)
    return pix2mm.to(device) @ pts   # (4,P) homogeneous mm

def to_world(T, pts_mm):
    """T:(...,4,4), pts_mm:(4,P) -> (...,3,P)."""
    return (T @ pts_mm)[..., :3, :]

def pose_point_loss(pred_rel, gt_rel, pts_mm):
    """Mean squared point distance between pred and GT relative transforms."""
    pw = to_world(pred_rel, pts_mm); gw = to_world(gt_rel, pts_mm)
    return ((pw - gw) ** 2).sum(dim=1).mean()

def reconstruction_errors(pred_rel, gt_rel, pix2mm, H, W, stride=8, landmarks=None):
    """Official 4-metric set in mm: {global,local} x {all-pixel,landmark}. 0 when pred==gt.
    landmarks: (K,3) tensor [frame_pos, x_px, y_px] or None -> landmark keys stay None."""
    pts = frame_points(H, W, pix2mm, stride=stride, device=pred_rel.device)
    gp, lp = accumulate_global(pred_rel), accumulate_local(pred_rel, 1)
    gg, lg = accumulate_global(gt_rel), accumulate_local(gt_rel, 1)
    def mean_all(a, b):
        d = to_world(a, pts) - to_world(b, pts)          # (N,3,P)
        return torch.sqrt((d ** 2).sum(dim=1)).mean().item()
    out = {"global_all": mean_all(gp, gg), "local_all": mean_all(lp, lg),
           "global_landmark": None, "local_landmark": None}
    if landmarks is not None and len(landmarks) > 0:
        lm = landmarks.to(pred_rel.device).float()
        pos = lm[:, 0].long().clamp(0, gp.shape[0] - 1)
        ones = torch.ones(lm.shape[0], device=lm.device)
        q = pix2mm @ torch.stack([lm[:, 1], lm[:, 2], torch.zeros_like(ones), ones], 0)  # (4,K)
        def lm_err(Gp, Gg):
            wp = torch.matmul(Gp[pos], q.T.unsqueeze(-1)).squeeze(-1)[:, :3]  # (K,3)
            wg = torch.matmul(Gg[pos], q.T.unsqueeze(-1)).squeeze(-1)[:, :3]
            return torch.sqrt(((wp - wg) ** 2).sum(-1)).mean().item()
        out["global_landmark"] = lm_err(gp, gg)
        out["local_landmark"] = lm_err(lp, lg)
    return out

print("geometry + metrics defined")

## 4. Real-data reader + frame preprocessing

In [ ]:
import numpy as np, h5py, torch, torch.nn.functional as F

def list_scans(root, subjects):
    items = []
    for s in subjects:
        for p in sorted((Path(root) / s).glob("*.h5")):
            items.append((s, p))
    return items

def read_scan(path):
    with h5py.File(path, "r") as f:
        frames = f[FRAMES_KEY][()]
        tforms = f[TFORMS_KEY][()]
    frames = np.asarray(frames)
    tforms = np.asarray(tforms).astype(np.float32)
    if tforms.ndim == 2 and tforms.shape[-1] == 16:      # (N,16) -> (N,4,4)
        tforms = tforms.reshape(-1, 4, 4)
    return frames, tforms

def prep_frames(frames_np, idx, image_size, device):
    x = torch.from_numpy(frames_np[idx].astype(np.float32))   # (n,H,W)
    x = x.unsqueeze(1)                                        # (n,1,H,W)
    x = F.interpolate(x, size=(image_size, image_size), mode="bilinear", align_corners=False)
    x = (x - x.mean()) / (x.std() + 1e-6)
    return x.to(device)

def load_scan_landmarks(subject, scan_stem, idx):
    """Load landmarks for one scan and map their frame index onto the eval subsample `idx`.
    Assumes landmark_XXX.h5 keyed by scan stem, array (K,>=3) = [frame_idx, x_px, y_px].
    Returns (K,3) tensor [pos_in_idx, x, y] or None. Provisional schema — confirm on real files."""
    if not LANDMARK_DIR:
        return None
    try:
        sidx = int(str(subject).replace("subject", ""))
    except ValueError:
        return None
    f = Path(LANDMARK_DIR) / f"landmark_{sidx:03d}.h5"
    if not f.exists():
        return None
    with h5py.File(f, "r") as h:
        if scan_stem not in h:
            return None
        arr = np.asarray(h[scan_stem][()], dtype=float)
    if arr.ndim == 1:
        arr = arr[None, :]
    if arr.shape[1] < 3:
        return None
    idx_arr = np.asarray(idx)
    pos = np.array([int(np.argmin(np.abs(idx_arr - fo))) for fo in arr[:, 0]])
    return torch.from_numpy(np.stack([pos, arr[:, 1], arr[:, 2]], axis=1)).float()

all_items = list_scans(DATA_ROOT, subjects)
print(f"{len(all_items)} scans across {len(subjects)} subjects")
fr0, tf0 = read_scan(all_items[0][1])
print("example scan:", all_items[0][1].name, "| frames", fr0.shape, "| tforms", tf0.shape)

## 5. Build model + subject-level train/val split

Split is by **subject** (not by scan) so validation measures generalization to unseen anatomy.
Encoder + pose head are trained end-to-end here (no SSL pretraining available from scratch).

### Calibration (`calib_matrix.csv`)

If `CALIB_PATH` points at the official `calib_matrix.csv`, the pixel-placement matrix becomes the real
`C @ S` — anisotropic pixel→mm scaling **and** the image→tracking-tool rigid offset (~114 mm lever
arm) — so all mm metrics, the loss, and the reconstructions are properly calibrated. Without it, the
notebook falls back to an isotropic `PIXEL_SPACING` guess (indicative, not leaderboard-comparable).
The same `pix2mm` flows into training, evaluation, compounding, and the INR — no other cell changes.

In [ ]:
import random
from usrecon.encoders import build_encoder
from usrecon.pose import build_pose_regressor

random.seed(SEED)
subs = subjects[:]
random.shuffle(subs)
n_val = min(MAX_VAL_SUBJECTS, max(1, len(subs) // 5))
val_subjects   = subs[:n_val]
train_subjects = subs[n_val:n_val + MAX_TRAIN_SUBJECTS]
print("train subjects:", train_subjects)
print("val subjects  :", val_subjects)

train_items = list_scans(DATA_ROOT, train_subjects)
val_items   = list_scans(DATA_ROOT, val_subjects)
print(f"{len(train_items)} train scans | {len(val_items)} val scans")

enc_cfg = {"type": ENCODER_TYPE, "image_size": IMAGE_SIZE, "patch_size": 16,
           "in_chans": 1, "embed_dim": EMBED_DIM, "depth": 4, "num_heads": 4, "k": 9}
encoder = build_encoder(enc_cfg).to(DEVICE)
pose    = build_pose_regressor({"embed_dim": EMBED_DIM}).to(DEVICE)

pix2mm, CALIB_MSG, IS_CALIBRATED = build_pixel_to_tool(CALIB_PATH, PIXEL_SPACING, DEVICE)
print(CALIB_MSG)
loss_pts = frame_points(IMAGE_SIZE, IMAGE_SIZE, pix2mm, stride=None, device=DEVICE)  # corners+center
n_params = sum(p.numel() for p in list(encoder.parameters()) + list(pose.parameters()))
print(f"model ready: {n_params/1e6:.2f}M params")

## 6. Train

In [ ]:
import random, math
from usrecon.utils.viz import _fig_path
import matplotlib.pyplot as plt

opt = torch.optim.Adam(list(encoder.parameters()) + list(pose.parameters()), lr=LR)
encoder.train(); pose.train()
epoch_losses = []

for epoch in range(EPOCHS):
    random.shuffle(train_items)
    running, nseen = 0.0, 0
    for s, path in train_items:
        frames_np, tforms_np = read_scan(path)
        n = len(frames_np)
        if n < 2:
            continue
        w = min(WINDOW, n)
        n0 = random.randint(0, n - w)
        idx = list(range(n0, n0 + w))
        frames = prep_frames(frames_np, idx, IMAGE_SIZE, DEVICE)     # (w,1,H,W)
        emb = encoder(frames).unsqueeze(0)                          # (1,w,D)
        pred7 = pose(emb)[0]                                        # (w,7)
        pred_rel = quat_trans_to_matrix(pred7)                      # (w,4,4)
        tfw = torch.from_numpy(tforms_np[idx]).to(DEVICE)
        gt_rel = relative_from_absolute(tfw)
        loss = pose_point_loss(pred_rel, gt_rel, loss_pts)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item(); nseen += 1
    epoch_losses.append(running / max(1, nseen))
    print(f"epoch {epoch+1}/{EPOCHS}  point-loss(mm^2) = {epoch_losses[-1]:.4f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(1, len(epoch_losses)+1), epoch_losses, marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("point-loss (mm^2)"); ax.set_title("training loss")
fig.tight_layout(); fig.savefig(_fig_path("real_train", "loss_curve"), dpi=120); plt.show()

## 7. Evaluate + validate (reconstruction error, mm)

For each scan: predict per-frame relative transforms, accumulate them into a trajectory, and compare the
reconstructed image-point positions against the GT `tforms`. **Global** shows accumulated drift; **Local**
is per-step consistency. Reported as the mean over the split's scans.

In [ ]:
import numpy as np

@torch.no_grad()
def evaluate(items, label):
    encoder.eval(); pose.eval()
    per_scan = []
    traj_example = None
    for s, path in items:
        frames_np, tforms_np = read_scan(path)
        n = len(frames_np)
        if n < 2:
            continue
        # subsample long scans to bound cost
        if n > MAX_EVAL_FRAMES:
            idx = list(np.linspace(0, n - 1, MAX_EVAL_FRAMES).astype(int))
        else:
            idx = list(range(n))
        frames = prep_frames(frames_np, idx, IMAGE_SIZE, DEVICE)
        emb = encoder(frames).unsqueeze(0)
        pred_rel = quat_trans_to_matrix(pose(emb)[0])
        tfw = torch.from_numpy(tforms_np[idx]).to(DEVICE)
        gt_rel = relative_from_absolute(tfw)
        lm = load_scan_landmarks(s, path.stem, idx)
        m = reconstruction_errors(pred_rel, gt_rel, pix2mm, IMAGE_SIZE, IMAGE_SIZE,
                                  stride=METRIC_STRIDE, landmarks=lm)
        per_scan.append(m)
        if traj_example is None:
            gp = accumulate_global(pred_rel)[:, :2, 3].cpu().numpy()
            gg = accumulate_global(gt_rel)[:, :2, 3].cpu().numpy()
            traj_example = (gp, gg)
    keys = ("global_all", "local_all", "global_landmark", "local_landmark")
    agg = {}
    for k in keys:
        vals = [m[k] for m in per_scan if m.get(k) is not None]
        agg[k] = float(np.mean(vals)) if vals else None
    def _f(k):
        return f"{agg[k]:.3f}" if agg[k] is not None else "n/a"
    print(f"[{label}] scans={len(per_scan)}  global_all={_f('global_all')}  local_all={_f('local_all')}  "
          f"global_lm={_f('global_landmark')}  local_lm={_f('local_landmark')}  (mm)")
    return agg, per_scan, traj_example

train_metrics, train_per, _        = evaluate(train_items, "train")
val_metrics,   val_per,  val_traj  = evaluate(val_items,   "val")

## 8. Figures + run manifest

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
from usrecon.utils.viz import _fig_path
from usrecon.paths import OUTPUT_DIR

# error histogram (per-scan global-all on val)
vals = [m["global_all"] for m in val_per] or [0.0]
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(vals, bins=min(20, max(3, len(vals))))
ax.set_xlabel("per-scan global all-pixel error (mm)"); ax.set_ylabel("count")
ax.set_title("validation reconstruction error")
fig.tight_layout(); fig.savefig(_fig_path("real_eval", "val_error_hist"), dpi=120); plt.show()

# trajectory: predicted vs GT (first val scan)
if val_traj is not None:
    gp, gg = val_traj
    drift = np.linalg.norm(gp - gg, axis=1)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    axs[0].plot(gg[:, 0], gg[:, 1], "-o", ms=3, label="GT")
    axs[0].plot(gp[:, 0], gp[:, 1], "-x", ms=3, label="pred")
    axs[0].set_title("probe trajectory (x,y mm)"); axs[0].legend(); axs[0].axis("equal")
    axs[1].plot(range(1, len(drift)+1), drift, marker=".")
    axs[1].set_xlabel("frame"); axs[1].set_ylabel("drift (mm)"); axs[1].set_title("cumulative drift")
    fig.tight_layout(); fig.savefig(_fig_path("real_eval", "val_trajectory"), dpi=120); plt.show()

# save model + manifest
ckpt_dir = OUTPUT_DIR / "real_train"; ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save({"encoder": encoder.state_dict(), "pose": pose.state_dict()}, ckpt_dir / "model.pt")
manifest = {
    "status": "pass",
    "encoder": ENCODER_TYPE, "image_size": IMAGE_SIZE, "epochs": EPOCHS,
    "train_subjects": train_subjects, "val_subjects": val_subjects,
    "train_metrics_mm": train_metrics, "val_metrics_mm": val_metrics,
    "final_train_loss": epoch_losses[-1] if epoch_losses else None,
    "pixel_spacing_mm": PIXEL_SPACING, "calibrated": bool(IS_CALIBRATED),
    "calibration_note": CALIB_MSG,
}
(ckpt_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## Notes / next steps

- **Scale up** by raising `MAX_TRAIN_SUBJECTS`, `EPOCHS`, `IMAGE_SIZE`, and `WINDOW`. The defaults are sized
  for a quick end-to-end pass, not leaderboard accuracy.
- **Exact challenge units:** attach the official calibration matrix and set `CALIB_PATH` to replace the
  approximate `PIXEL_SPACING` scale, then add landmark files to enable the landmark reconstruction errors.
- **Frozen-encoder variant:** to match the design's SSL-pretrain-then-freeze recipe, load a pretrained
  encoder checkpoint and pass only `pose.parameters()` to the optimizer.

## 9. Reconstruction helpers

Compound a scan's frames into a 3D point cloud using a set of per-frame **global** transforms
(each frame's pixels placed into the frame-0 world), then voxelize for slice views. Used by the
sample visualization, the save-all-reconstructions pass, and the contribution analysis below.

In [ ]:
import numpy as np, torch, torch.nn.functional as F

def predict_scan(frames_np, tforms_np, idx):
    """-> (pred_rel, gt_rel) relative transforms (len(idx),4,4) on DEVICE."""
    frames = prep_frames(frames_np, idx, IMAGE_SIZE, DEVICE)
    with torch.no_grad():
        pred_rel = quat_trans_to_matrix(pose(encoder(frames).unsqueeze(0))[0])
    gt_rel = relative_from_absolute(torch.from_numpy(tforms_np[idx]).float().to(DEVICE))
    return pred_rel, gt_rel

def eval_frame_indices(n):
    if n > MAX_EVAL_FRAMES:
        return list(np.linspace(0, n - 1, MAX_EVAL_FRAMES).astype(int))
    return list(range(n))

def compound_points(frames_np, idx, global_T, image_size, pix2mm, stride=4):
    """Place each frame's (subsampled) pixels into frame-0 world mm.
    -> points (P,3), intensity (P,) as numpy."""
    fr = torch.from_numpy(frames_np[idx].astype('float32')).unsqueeze(1)
    fr = F.interpolate(fr, size=(image_size, image_size), mode='bilinear', align_corners=False)[:, 0]
    H = W = image_size
    gu = torch.arange(0, W, stride).float(); gv = torch.arange(0, H, stride).float()
    uu, vv = torch.meshgrid(gu, gv, indexing='xy')
    us = uu.reshape(-1); vs = vv.reshape(-1)
    pts_pix = torch.stack([us, vs, torch.zeros_like(us), torch.ones_like(us)], 0).to(global_T.device)
    pts_mm = pix2mm.to(global_T.device) @ pts_pix                      # (4,P)
    ui = us.long(); vi = vs.long()
    world = torch.matmul(global_T, pts_mm)[:, :3, :]                   # (n,3,P)
    inten = fr[:, vi, ui]                                              # (n,P)
    pts = world.permute(0, 2, 1).reshape(-1, 3).cpu().numpy()
    return pts, inten.reshape(-1).cpu().numpy()

def voxelize(points, intensity, res=64):
    mn = points.min(0); mx = points.max(0); span = mx - mn; span[span == 0] = 1.0
    ij = np.clip(((points - mn) / span * (res - 1)).astype(int), 0, res - 1)
    vol = np.zeros((res, res, res), np.float32); cnt = np.zeros((res, res, res), np.float32)
    np.add.at(vol, (ij[:, 0], ij[:, 1], ij[:, 2]), intensity)
    np.add.at(cnt, (ij[:, 0], ij[:, 1], ij[:, 2]), 1.0)
    cnt[cnt == 0] = 1.0
    return vol / cnt

print('reconstruction helpers ready')

## 10. Sample reconstruction + visualization

Reconstructs one scan with the **predicted** trajectory and, alongside it, the **GT** trajectory.
Left-to-right: three orthogonal max-intensity slices of the voxelized volume; last panel is the 3D
point cloud. If the prediction is good the two rows look alike; drift shows up as the predicted
volume smearing or curling away from the GT one.

In [ ]:
import matplotlib.pyplot as plt
from usrecon.utils.viz import _fig_path

def visualize_scan(frames_np, tforms_np, title, res=48, point_stride=4, max_scatter=4000):
    idx = eval_frame_indices(len(frames_np))
    pred_rel, gt_rel = predict_scan(frames_np, tforms_np, idx)
    gP, gG = accumulate_global(pred_rel), accumulate_global(gt_rel)
    pp, pi = compound_points(frames_np, idx, gP, IMAGE_SIZE, pix2mm, stride=point_stride)
    gp, gi = compound_points(frames_np, idx, gG, IMAGE_SIZE, pix2mm, stride=point_stride)
    vP, vG = voxelize(pp, pi, res), voxelize(gp, gi, res)

    fig = plt.figure(figsize=(16, 6))
    for row, (vol, pts, inten, name) in enumerate([(vP, pp, pi, 'PRED'), (vG, gp, gi, 'GT')]):
        for c, ax_i in enumerate([0, 1, 2]):
            ax = fig.add_subplot(2, 4, row * 4 + c + 1)
            ax.imshow(vol.max(axis=ax_i), cmap='gray'); ax.set_xticks([]); ax.set_yticks([])
            ax.set_ylabel(name if c == 0 else ''); ax.set_title(['axial', 'coronal', 'sagittal'][c] if row == 0 else '')
        ax3 = fig.add_subplot(2, 4, row * 4 + 4, projection='3d')
        sub = np.random.choice(len(pts), min(max_scatter, len(pts)), replace=False)
        ax3.scatter(pts[sub, 0], pts[sub, 1], pts[sub, 2], c=inten[sub], cmap='gray', s=1)
        ax3.set_title(f'{name} cloud'); ax3.set_xticks([]); ax3.set_yticks([]); ax3.set_zticks([])
    fig.suptitle(f'Reconstruction: {title}')
    fig.tight_layout()
    fig.savefig(_fig_path('real_eval', f'sample_reconstruction'), dpi=110); plt.show()

# reconstruct the first validation scan
_s, _p = val_items[0]
_fr, _tf = read_scan(_p)
visualize_scan(_fr, _tf, title=f'{_s}/{_p.stem}')

## 11. Save all reconstructions during evaluation

For every scan in a split, saves a compressed `.npz` (predicted + GT global transforms, compounded
point cloud + intensities, and the scan's metrics) plus a slice-preview PNG, under
`outputs/real_eval/reconstructions/`. An `index.json` lists every scan with its errors.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from usrecon.paths import OUTPUT_DIR

RECON_DIR = OUTPUT_DIR / 'real_eval' / 'reconstructions'
RECON_DIR.mkdir(parents=True, exist_ok=True)

def save_all_reconstructions(items, split, point_stride=4, res=48, max_points=40000):
    encoder.eval(); pose.eval()
    index = []
    for s, path in items:
        fr, tf = read_scan(path)
        if len(fr) < 2:
            continue
        idx = eval_frame_indices(len(fr))
        pred_rel, gt_rel = predict_scan(fr, tf, idx)
        gP, gG = accumulate_global(pred_rel), accumulate_global(gt_rel)
        pp, pi = compound_points(fr, idx, gP, IMAGE_SIZE, pix2mm, stride=point_stride)
        gp, gi = compound_points(fr, idx, gG, IMAGE_SIZE, pix2mm, stride=point_stride)
        if len(pp) > max_points:
            keep = np.random.choice(len(pp), max_points, replace=False)
            pp, pi = pp[keep], pi[keep]; gp, gi = gp[keep], gi[keep]
        m = reconstruction_errors(pred_rel, gt_rel, pix2mm, IMAGE_SIZE, IMAGE_SIZE, stride=METRIC_STRIDE)
        name = f'{split}__{s}__{path.stem}'
        np.savez_compressed(RECON_DIR / f'{name}.npz',
                            pred_global=gP.cpu().numpy(), gt_global=gG.cpu().numpy(),
                            pred_points=pp.astype(np.float32), pred_intensity=pi.astype(np.float32),
                            gt_points=gp.astype(np.float32), gt_intensity=gi.astype(np.float32),
                            frame_indices=np.array(idx),
                            **{k: (np.nan if m[k] is None else m[k]) for k in m})
        # quick slice preview
        vP = voxelize(pp, pi, res)
        fig, axs = plt.subplots(1, 3, figsize=(9, 3))
        for c in range(3):
            axs[c].imshow(vP.max(axis=c), cmap='gray'); axs[c].axis('off')
        fig.suptitle(f"{name}  global={m['global_all']:.1f}mm local={m['local_all']:.1f}mm")
        fig.tight_layout(); fig.savefig(RECON_DIR / f'{name}.png', dpi=90); plt.close(fig)
        index.append({'scan': name, **m})
    (RECON_DIR / f'index_{split}.json').write_text(json.dumps(index, indent=2))
    print(f'saved {len(index)} reconstructions for split={split} -> {RECON_DIR}')
    return index

# save reconstructions for validation scans (add train_items too if you want them all)
val_recon_index = save_all_reconstructions(val_items, 'val')

## 12. Component / contribution analysis

Attributes the reconstruction error to its parts by swapping in **baseline predictors** and
**per-DOF swaps**, all scored with the same metric:

- **GT oracle** — must be ~0 (sanity check on the whole pipeline).
- **identity (no motion)** — predict zero motion. If the trained model's *local* error isn't clearly
  below this, the model hasn't learned useful frame-to-frame motion (it's near-degenerate).
- **const mean translation** — predict the dataset's average per-step translation. A stronger trivial
  baseline than identity for straight sweeps.
- **trained model** — what you trained.
- **model rot + GT trans** / **GT rot + model trans** — isolates whether **rotation** or **translation**
  error drives the global drift (rotation errors compound fastest).

In [ ]:
import numpy as np, torch

# dataset-average per-step translation (from train GT) for the 'const mean translation' baseline
def mean_step_translation(items, max_scans=20):
    ts = []
    for s, path in items[:max_scans]:
        fr, tf = read_scan(path)
        if len(fr) < 2:
            continue
        rel = relative_from_absolute(torch.from_numpy(tf).float())
        ts.append(rel[1:, :3, 3].mean(0))
    return torch.stack(ts).mean(0).to(DEVICE) if ts else torch.zeros(3, device=DEVICE)

MEAN_T = mean_step_translation(train_items)

def const_trans_rel(n, device):
    rel = torch.eye(4, device=device).repeat(n, 1, 1)
    rel[1:, :3, 3] = MEAN_T
    return rel

def swap_dof(pred, gt, rot_from):
    out = gt.clone()
    if rot_from == 'pred':
        out[:, :3, :3] = pred[:, :3, :3]; out[:, :3, 3] = gt[:, :3, 3]
    else:  # translation from pred, rotation from gt
        out[:, :3, :3] = gt[:, :3, :3]; out[:, :3, 3] = pred[:, :3, 3]
    return out

PREDICTORS = {
    'GT oracle':             lambda p, g: g,
    'identity (no motion)':  lambda p, g: torch.eye(4, device=g.device).repeat(g.shape[0], 1, 1),
    'const mean translation':lambda p, g: const_trans_rel(g.shape[0], g.device),
    'trained model':         lambda p, g: p,
    'model rot + GT trans':  lambda p, g: swap_dof(p, g, 'pred'),
    'GT rot + model trans':  lambda p, g: swap_dof(p, g, 'trans'),
}

def contribution_table(items, label):
    encoder.eval(); pose.eval()
    acc = {name: {'global_all': [], 'local_all': []} for name in PREDICTORS}
    for s, path in items:
        fr, tf = read_scan(path)
        if len(fr) < 2:
            continue
        idx = eval_frame_indices(len(fr))
        pred_rel, gt_rel = predict_scan(fr, tf, idx)
        for name, fn in PREDICTORS.items():
            m = reconstruction_errors(fn(pred_rel, gt_rel), gt_rel, pix2mm,
                                      IMAGE_SIZE, IMAGE_SIZE, stride=METRIC_STRIDE)
            acc[name]['global_all'].append(m['global_all'])
            acc[name]['local_all'].append(m['local_all'])
    print(f'\n=== contribution analysis [{label}] ===')
    print(f"{'predictor':<24}{'global_all(mm)':>16}{'local_all(mm)':>16}")
    print('-' * 56)
    for name in PREDICTORS:
        g = float(np.mean(acc[name]['global_all'])); l = float(np.mean(acc[name]['local_all']))
        print(f'{name:<24}{g:>16.3f}{l:>16.3f}')
    return acc

_ = contribution_table(val_items, 'val')

### How to read the contribution table

- **trained model local vs identity local:** the model's real learned signal. If
  `trained < identity`, the encoder+pose head genuinely predict inter-frame motion; if not, they don't yet.
- **trained global vs const-mean-translation global:** whether the model beats a dumb constant-velocity
  guess once errors accumulate.
- **model rot + GT trans vs GT rot + model trans:** whichever is *worse* than the other is the DOF hurting
  you most. If `model rot + GT trans` blows up while `GT rot + model trans` stays low, **rotation error is
  the drift driver** — the usual culprit, and the argument for adding rotation-aware losses or drift
  correction next.

## 13. 3D rotating GIF of the reconstruction

Spins the compounded 3D point cloud through 360° and saves an animated GIF (displayed inline). Renders
the **predicted** volume and a **PRED-vs-GT** side-by-side so drift is visible as the predicted cloud
twisting away from the ground-truth shape. GIFs are written to `outputs/real_eval/`.

In [ ]:
import io, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as IPyImage, display
from usrecon.paths import OUTPUT_DIR

GIF_DIR = OUTPUT_DIR / 'real_eval'; GIF_DIR.mkdir(parents=True, exist_ok=True)

def _equal_3d(ax, pts):
    c = pts.mean(0); r = float((pts.max(0) - pts.min(0)).max()) / 2 + 1e-6
    ax.set_xlim(c[0]-r, c[0]+r); ax.set_ylim(c[1]-r, c[1]+r); ax.set_zlim(c[2]-r, c[2]+r)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

def _fig_to_pil(fig):
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=85); plt.close(fig)
    buf.seek(0); return Image.open(buf).convert('RGB')

def _subsample(pts, inten, max_pts):
    if len(pts) > max_pts:
        k = np.random.choice(len(pts), max_pts, replace=False); return pts[k], inten[k]
    return pts, inten

def rotating_gif(pts, inten, out_path, title='', n_frames=36, elev=18, fps=12, s=1.5, max_pts=6000):
    pts, inten = _subsample(np.asarray(pts), np.asarray(inten), max_pts)
    frames = []
    for az in np.linspace(0, 360, n_frames, endpoint=False):
        fig = plt.figure(figsize=(4.5, 4.5)); ax = fig.add_subplot(projection='3d')
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=inten, cmap='gray', s=s, linewidths=0)
        ax.view_init(elev=elev, azim=az); _equal_3d(ax, pts); ax.set_title(title, fontsize=10)
        fig.tight_layout(); frames.append(_fig_to_pil(fig))
    frames[0].save(out_path, save_all=True, append_images=frames[1:],
                   duration=int(1000 / fps), loop=0)
    return out_path

def rotating_pair_gif(pp, pi, gp, gi, out_path, title='', n_frames=36, elev=18, fps=12, s=1.5, max_pts=6000):
    pp, pi = _subsample(np.asarray(pp), np.asarray(pi), max_pts)
    gp, gi = _subsample(np.asarray(gp), np.asarray(gi), max_pts)
    frames = []
    for az in np.linspace(0, 360, n_frames, endpoint=False):
        fig = plt.figure(figsize=(9, 4.5))
        for j, (P, I, name) in enumerate([(pp, pi, 'PRED'), (gp, gi, 'GT')]):
            ax = fig.add_subplot(1, 2, j + 1, projection='3d')
            ax.scatter(P[:, 0], P[:, 1], P[:, 2], c=I, cmap='gray', s=s, linewidths=0)
            ax.view_init(elev=elev, azim=az); _equal_3d(ax, P); ax.set_title(name, fontsize=11)
        fig.suptitle(title, fontsize=11); fig.tight_layout(); frames.append(_fig_to_pil(fig))
    frames[0].save(out_path, save_all=True, append_images=frames[1:],
                   duration=int(1000 / fps), loop=0)
    return out_path

# --- build a reconstruction for one validation scan and animate it ---
_s, _p = val_items[0]
_fr, _tf = read_scan(_p)
_idx = eval_frame_indices(len(_fr))
_pred_rel, _gt_rel = predict_scan(_fr, _tf, _idx)
_gP, _gG = accumulate_global(_pred_rel), accumulate_global(_gt_rel)
_pp, _pi = compound_points(_fr, _idx, _gP, IMAGE_SIZE, pix2mm, stride=3)
_gp, _gi = compound_points(_fr, _idx, _gG, IMAGE_SIZE, pix2mm, stride=3)

pred_gif = rotating_gif(_pp, _pi, GIF_DIR / 'pred_reconstruction_3d.gif', title=f'PRED {_s}/{_p.stem}')
pair_gif = rotating_pair_gif(_pp, _pi, _gp, _gi, GIF_DIR / 'pred_vs_gt_3d.gif', title=f'{_s}/{_p.stem}')
print('saved:', pred_gif.name, '|', pair_gif.name)
display(IPyImage(filename=str(pair_gif)))

## 14. Interactive 3D (optional, Plotly)

A drag-to-rotate view in the notebook output. Heavier than the GIF; subsample hard if it lags. Self-skips
if Plotly isn't installed.

In [ ]:
try:
    import plotly.graph_objects as go
    import numpy as np
    pp_s, pi_s = _subsample(_pp, _pi, 8000)
    fig = go.Figure(data=[go.Scatter3d(
        x=pp_s[:, 0], y=pp_s[:, 1], z=pp_s[:, 2], mode='markers',
        marker=dict(size=1.5, color=pi_s, colorscale='Gray', opacity=0.7))])
    fig.update_layout(title=f'Predicted 3D reconstruction: {_s}/{_p.stem}',
                      scene=dict(aspectmode='data'), height=600, margin=dict(l=0, r=0, t=30, b=0))
    fig.show()
except Exception as e:
    print('[skip] interactive 3D (plotly) unavailable:', repr(e))

# Stage 2 → 3 → 4a: from a sparse cloud to a continuous implicit volume

This is the pipeline's core claim — the **implicit neural representation (INR)** turns the sparse,
gap-ridden compounded point cloud into a *continuous, arbitrary-resolution* volume (the “jelly”). The
cells below run the real Stage 2→3→4a chain on one inferred scan and visualize what each stage does to
the data. **No pipeline logic is changed** — we reuse `ImplicitFieldRegressor` from the package and only
add visualization.

**Stage 2 — compounding** places each frame's pixels into world space using the inferred pose:

$$ p_{\text{world}} = T_i \,\big(s\,[u,\,v,\,0,\,1]^\top\big), \qquad s=\text{pixel spacing} $$

**Stage 3 — implicit field** fits a coordinate-MLP $f_\theta:\mathbb{R}^3\!\to\!\mathbb{R}$ to those samples,
with a Fourier feature encoding to beat spectral bias:

$$ \gamma(p) = \big[\,p,\ \sin(2^0\pi p),\cos(2^0\pi p),\dots,\sin(2^{L-1}\pi p),\cos(2^{L-1}\pi p)\,\big] $$
$$ \mathcal{L}_{\text{recon}} = \sum_i \big\lVert f_\theta(p_i) - I_i \big\rVert^2 $$

**Stage 4a — render** densely samples the *frozen* $f_\theta$ on a grid at any resolution. Because
$f_\theta$ is continuous, the same field can be queried at 24³ or 96³ and stays smooth — that is the
“lively / flexible” property a fixed voxel grid does not have.

In [ ]:
# ---- INR settings (visualization only; tune freely) ----
INR_STEPS     = 500      # fit iterations for the implicit field on one scan
INR_HIDDEN    = 128
INR_LAYERS    = 4
INR_FREQS     = 6        # Fourier feature bands (L)
INR_BATCH     = 8192     # point minibatch per step
INR_GRID_RES  = 64       # render grid resolution for Stage 4a
INR_POINT_STRIDE = 2     # denser cloud for a richer field

import numpy as np, torch, torch.nn.functional as F
from usrecon.reconstruction import ImplicitFieldRegressor

def normalize_cloud(pts):
    """(P,3) world mm -> [-1,1] cube; returns normalized pts + (center, scale)."""
    c = pts.mean(0); r = float(np.abs(pts - c).max()) + 1e-6
    return (pts - c) / r, (c, r)

def make_grid(res, device):
    xs = torch.linspace(-1, 1, res)
    gx, gy, gz = torch.meshgrid(xs, xs, xs, indexing='ij')
    return torch.stack([gx.reshape(-1), gy.reshape(-1), gz.reshape(-1)], -1).to(device)

@torch.no_grad()
def render_field(field, res, device, chunk=200000):
    grid = make_grid(res, device); out = []
    for i in range(0, grid.shape[0], chunk):
        out.append(field(grid[i:i+chunk][None])[0, :, 0])
    return torch.cat(out).reshape(res, res, res).cpu().numpy()

def fit_implicit_field(pts_norm, inten, steps, snapshot_at=(), grid_res=48):
    """Stage 3: fit f_theta to (points, intensity). Returns field, losses, {step: volume}."""
    P = torch.from_numpy(pts_norm).float().to(DEVICE)
    I = torch.from_numpy(inten).float().to(DEVICE)
    I = (I - I.min()) / (I.max() - I.min() + 1e-6)          # normalize intensity to [0,1]
    field = ImplicitFieldRegressor(dim_in=3, hidden_dim=INR_HIDDEN, num_layers=INR_LAYERS,
                                   positional_encoding='fourier', pe_num_freqs=INR_FREQS).to(DEVICE)
    opt = torch.optim.Adam(field.parameters(), lr=1e-3)
    losses, snaps = [], {}
    n = P.shape[0]
    for step in range(steps):
        sel = torch.randint(0, n, (min(INR_BATCH, n),), device=DEVICE)
        pred = field(P[sel][None])[0, :, 0]
        loss = F.mse_loss(pred, I[sel])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step in snapshot_at:
            snaps[step] = render_field(field, grid_res, DEVICE)
    return field, losses, snaps

print('INR fit/render helpers ready (reusing package ImplicitFieldRegressor)')

## Fit the implicit field on one inferred scan and show the stage-by-stage transformation

In [ ]:
import matplotlib.pyplot as plt
from usrecon.utils.viz import _fig_path

# --- Stage 2: compound one validation scan with the INFERRED (predicted) poses ---
s_, p_ = val_items[0]
fr_, tf_ = read_scan(p_)
idx_ = eval_frame_indices(len(fr_))
pred_rel_, _ = predict_scan(fr_, tf_, idx_)
gP_ = accumulate_global(pred_rel_)
pts_, inten_ = compound_points(fr_, idx_, gP_, IMAGE_SIZE, pix2mm, stride=INR_POINT_STRIDE)
pts_n, (cc_, rr_) = normalize_cloud(pts_)
print(f'Stage 2 cloud: {pts_.shape[0]} points from {len(idx_)} frames')

# --- Stage 3: fit the implicit field, snapshotting the volume as it converges ---
snap_steps = sorted(set([0, INR_STEPS//8, INR_STEPS//4, INR_STEPS//2, INR_STEPS-1]))
field_, losses_, snaps_ = fit_implicit_field(pts_n, inten_, INR_STEPS,
                                             snapshot_at=snap_steps, grid_res=INR_GRID_RES)
print(f'Stage 3 INR fit: final L_recon = {losses_[-1]:.5f}')

# --- Stage 4a: dense render of the frozen field ---
vol_inr = render_field(field_, INR_GRID_RES, DEVICE)

# --- Stage 2 sparse voxelization (nearest splat) for the before/after comparison ---
vol_sparse = voxelize(pts_n, (inten_ - inten_.min())/(inten_.ptp()+1e-6), res=INR_GRID_RES)

# ---- Stage-progression panel: frame -> Stage2 sparse -> Stage3 INR -> Stage4a MIP ----
mid = INR_GRID_RES // 2
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
axs[0].imshow(fr_[idx_[len(idx_)//2]], cmap='gray'); axs[0].set_title('input frame (Stage 0 in)')
axs[1].imshow(vol_sparse[:, :, mid], cmap='gray'); axs[1].set_title('Stage 2: compounded (sparse)')
axs[2].imshow(vol_inr[:, :, mid], cmap='gray'); axs[2].set_title('Stage 3: implicit field (continuous)')
axs[3].imshow(vol_inr.max(axis=2), cmap='gray'); axs[3].set_title('Stage 4a: render (MIP)')
for a in axs: a.axis('off')
fig.suptitle(f'Stage 2→3→4a transformation  —  {s_}/{p_.stem}')
fig.tight_layout(); fig.savefig(_fig_path('real_eval', 'stage_progression'), dpi=120); plt.show()

# ---- L_recon convergence curve ----
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses_); ax.set_yscale('log'); ax.set_xlabel('INR step'); ax.set_ylabel('L_recon (log)')
ax.set_title('Stage 3: implicit-field convergence')
fig.tight_layout(); fig.savefig(_fig_path('real_eval', 'inr_convergence'), dpi=120); plt.show()

## Research-grade GIFs: the field forming, arbitrary-resolution querying, and a volume fly-through

In [ ]:
import io, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as IPyImage, display
from usrecon.paths import OUTPUT_DIR
GIF_DIR = OUTPUT_DIR / 'real_eval'; GIF_DIR.mkdir(parents=True, exist_ok=True)

def _panel_to_pil(imgs, titles, suptitle):
    fig, axs = plt.subplots(1, len(imgs), figsize=(3*len(imgs), 3.2))
    axs = np.atleast_1d(axs)
    for a, im, t in zip(axs, imgs, titles):
        a.imshow(im, cmap='gray'); a.set_title(t, fontsize=10); a.axis('off')
    fig.suptitle(suptitle, fontsize=11); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=85); plt.close(fig)
    buf.seek(0); return Image.open(buf).convert('RGB')

def save_gif(frames, out_path, fps=8):
    frames[0].save(out_path, save_all=True, append_images=frames[1:], duration=int(1000/fps), loop=0)
    return out_path

# (1) Convergence GIF: the implicit volume forming across training steps
mid = INR_GRID_RES // 2
conv_frames = [_panel_to_pil([snaps_[s][:, :, mid], snaps_[s].max(axis=2)],
                             ['mid slice', 'MIP'], f'INR at step {s}') for s in sorted(snaps_)]
g1 = save_gif(conv_frames, GIF_DIR / 'inr_convergence.gif', fps=2)

# (2) Arbitrary-resolution GIF: same field queried at increasing grid resolution (stays smooth)
res_list = [16, 24, 32, 48, 64, 96]
res_frames = []
for r in res_list:
    v = render_field(field_, r, DEVICE)
    res_frames.append(_panel_to_pil([v[:, :, r//2], v.max(axis=2)], ['mid slice', 'MIP'],
                                    f'render @ {r}³ (one continuous field)'))
g2 = save_gif(res_frames, GIF_DIR / 'inr_resolution.gif', fps=2)

# (3) Fly-through GIF: sweep the slicing plane through the continuous volume
sweep = [_panel_to_pil([vol_inr[:, :, k]], [f'z = {k}/{INR_GRID_RES}'],
                       'Stage 4a fly-through (INR volume)') for k in range(0, INR_GRID_RES, 2)]
g3 = save_gif(sweep, GIF_DIR / 'inr_flythrough.gif', fps=12)

print('saved:', g1.name, g2.name, g3.name)
display(IPyImage(filename=str(g1)))
display(IPyImage(filename=str(g3)))

### What these show

- **`stage_progression`** — the same anatomy as raw frame → sparse compounded splat (holes between frames)
  → continuous implicit field (holes filled) → rendered MIP. This is the enhancement the INR provides.
- **`inr_convergence.gif`** — $f_\theta$ literally forming as $\mathcal{L}_{\text{recon}}$ falls; the volume
  sharpens from blur to structure.
- **`inr_resolution.gif`** — the *same* trained field sampled at 16³→96³. A voxel grid would look blocky;
  the continuous field stays smooth — the core “flexible / arbitrary-resolution” claim, made visual.
- **`inr_flythrough.gif`** — a plane swept through the reconstructed volume, i.e. novel cross-sections the
  original 2D sweep never acquired — queried straight out of $f_\theta$.